# Measurement API additions

This notebook shows examples for recently added classes:

- `MeasurementResult`
- `PulseScheduleSnapshot`
- `DeviceExecutor` / `QuelDeviceExecutor`


In [ ]:
from pathlib import Path

import numpy as np

from qubex.backend import DeviceExecutor, QuelDeviceExecutor
from qubex.measurement.models import MeasurementResult, PulseScheduleSnapshot

## 1) Build `MeasurementResult` directly

`MeasurementResult` is the canonical primitive result model.


In [ ]:
result = MeasurementResult(
    mode="avg",
    data={
        "Q00": [np.array([1.0 + 2.0j, 3.0 + 4.0j])],
        "Q01": [np.array([5.0 + 6.0j, 7.0 + 8.0j])],
    },
    device_config={"box": "example"},
    measurement_config={"mode": "avg", "shots": 1024},
    pulse_schedule=PulseScheduleSnapshot(
        schema_version=1,
        channel_labels=["RQ00", "RQ01"],
        total_duration=512.0,
        total_length=256,
        waveforms=None,  # Set when save_waveforms=True in Measurement.run/execute
    ),
)

result

## 2) Save and load with NetCDF

`MeasurementResult` supports NetCDF persistence.


In [ ]:
out = Path(".rawdata/example_measurement_result.nc")
out.parent.mkdir(exist_ok=True, parents=True)

saved_path = result.save_netcdf(out)
loaded = MeasurementResult.load_netcdf(saved_path)

print(saved_path)
print(loaded.mode, loaded.device_config)

## 3) Convert to legacy result types

Use these adapters while migrating from old APIs.


In [ ]:
multiple = result.to_multiple_measure_result()
single = result.to_measure_result(index=0)

print(type(multiple).__name__, type(single).__name__)

## 4) `DeviceExecutor` abstraction

`Measurement` depends on the `DeviceExecutor` interface.
`QuelDeviceExecutor` is the concrete implementation for QUEL hardware.


In [ ]:
# Type-level usage (example)
executor_type = DeviceExecutor
quel_executor_type = QuelDeviceExecutor

print(executor_type.__name__, quel_executor_type.__name__)

## 5) (Optional) include waveforms

When calling `Measurement.run(...)` / `Measurement.execute(...)`,
set `save_waveforms=True` if you want waveform snapshots stored in the result.
By default, this is `False` to keep output files smaller.
